In [1]:
import pandas as pd
import numpy as np
import sys
import os
import warnings

# --- Imports de Deep Learning (Keras/TensorFlow) ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Imports de Machine Learning (Sklearn) ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import resample # Para o Undersampling
from sklearn.exceptions import UndefinedMetricWarning

# --- Imports de Balanceamento ---
# (Precisamos dele para o SMOTE ou Undersampling)
!pip install -q imbalanced-learn
from imblearn.under_sampling import RandomUnderSampler

# Ignora warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Montar o Google Drive ---
print("Montando Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

Montando Google Drive...
Mounted at /content/drive


In [2]:
# Vamos confirmar que o Colab nos deu a GPU
print("Verificando GPU...")
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print(
      '\n\nATENÇÃO: GPU não encontrada! '
      'Vá em "Ambiente de execução" -> "Alterar o tipo de ambiente de execução" e selecione "GPU (T4)".'
  )
else:
  print(f'GPU encontrada: {device_name}')
  !nvidia-smi

Verificando GPU...
GPU encontrada: /device:GPU:0
Mon Nov  3 23:02:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             24W /   70W |     102MiB /  15360MiB |      2%      Default |
|                                         |                        |                  N/A |

In [3]:
# --- 1. Configurações ---
GDRIVE_PATH = '/content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/'
ARQUIVO_ENTRADA = os.path.join(GDRIVE_PATH, 'dataset/dataset_mestre_IMPUTADO_FINAL.csv')

TARGET_FUNCTION = 'protein binding'

# --- Configs da CNN ---
# Como os genes são de comprimentos MUITO variados (1k a 80k+),
# não podemos usar 'padding'. Vamos 'truncar' (cortar) todos
# em um comprimento fixo. 2000 é um bom começo.
MAX_SEQ_LENGTH = 2000
VOCAB_SIZE = 5 # A, T, C, G, e 'N' (embora a maioria tenha sido imputada)
EMBEDDING_DIM = 100 # Dimensão do vetor para cada base

# --- Configs do Pipeline ---
# Não podemos ler 14.4M de linhas. Vamos ler uma amostra.
# 500.000 linhas deve nos dar ~5k-10k genes únicos para um bom protótipo.
N_LINHAS_PARA_LER = 500000

In [5]:
print(f"Iniciando Classificação (CNN)...")
print(f"Lendo {N_LINHAS_PARA_LER} linhas de {ARQUIVO_ENTRADA}...")
print("Isso pode levar alguns minutos...")

# --- 3.1 Carregar Dados (Amostra) ---
try:
    df = pd.read_csv(ARQUIVO_ENTRADA, nrows=N_LINHAS_PARA_LER)
except FileNotFoundError:
    print(f"ERRO: Arquivo '{ARQUIVO_ENTRADA}' não encontrado no Drive.")
    raise

print("Dataset carregado.")

# --- 3.2 Preparação dos Labels (Y) ---
print(f"Preparando labels... (1 = '{TARGET_FUNCTION}', 0 = Outro)")
df['label'] = np.where(df['GO term name'] == TARGET_FUNCTION, 1, 0)

# --- 3.3 Achatamento (Handling Duplicates) ---
print("Achatando dataset (1 linha por gene)...")
df_labels_flat = df.groupby('join_key')['label'].max()
df_seq_flat = df.groupby('join_key')['Sequencia'].first()
# 'on=' cria um novo índice numérico 0-N, o que é esperado
df_flat = pd.merge(df_seq_flat, df_labels_flat, on='join_key', how='inner')
print(f"Dataset achatado para {len(df_flat)} genes únicos.")
print(f"Distribuição ANTES do balanceamento:\n{df_flat['label'].value_counts()}\n")

# --- 3.4 Balanceamento (Undersampling - Nossa melhor estratégia) ---
print("Iniciando Undersampling para forçar balanço 1:1...")
rus = RandomUnderSampler(random_state=42)

# X = os índices do df_flat (0, 1, 2...)
# y = os labels (1, 1, 0...)
X_para_amostrar = df_flat.index.values.reshape(-1, 1)
y_para_amostrar = df_flat['label']

X_res, y_res = rus.fit_resample(X_para_amostrar, y_para_amostrar)

# --- CORREÇÃO AQUI ---
# X_res contém os índices originais que queremos. Vamos achatá-lo.
indices_balanceados = X_res.flatten()
# --- FIM DA CORREÇÃO ---

df_balanced = df_flat.loc[indices_balanceados].copy()
print(f"Dataset balanceado criado.")
print(f"Distribuição DEPOIS do balanceamento:\n{df_balanced['label'].value_counts()}\n")

# --- 3.5 Preparação para CNN (Tokenizer e Padding) ---
print("Preparando dados para CNN (Tokenizing e Padding)...")

# 1. Converter "ATGC..." em "1 4 2 3..."
tokenizer = Tokenizer(char_level=True, lower=False)
tokenizer.fit_on_texts(df_balanced['Sequencia'])
sequences_tokenized = tokenizer.texts_to_sequences(df_balanced['Sequencia'])

# 2. Aplicar Padding/Truncating
X_padded = pad_sequences(sequences_tokenized,
                         maxlen=MAX_SEQ_LENGTH,
                         padding='post',
                         truncating='post')

Y_labels = df_balanced['label'].values

print(f"Matriz de features X criada: {X_padded.shape}")
print(f"Vetor de labels Y criado: {Y_labels.shape}")

# --- 3.6 Divisão de Treino/Teste ---
print("\nDividindo dados (80% treino / 20% teste)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    Y_labels,
    test_size=0.2,
    random_state=42,
    stratify=Y_labels
)

Iniciando Classificação (CNN)...
Lendo 500000 linhas de /content/drive/MyDrive/Faculdade/IC-2025.2/Biogenetica/dataset/dataset_mestre_IMPUTADO_FINAL.csv...
Isso pode levar alguns minutos...
Dataset carregado.
Preparando labels... (1 = 'protein binding', 0 = Outro)
Achatando dataset (1 linha por gene)...
Dataset achatado para 3253 genes únicos.
Distribuição ANTES do balanceamento:
label
1    2571
0     682
Name: count, dtype: int64

Iniciando Undersampling para forçar balanço 1:1...
Dataset balanceado criado.
Distribuição DEPOIS do balanceamento:
label
0    682
1    682
Name: count, dtype: int64

Preparando dados para CNN (Tokenizing e Padding)...
Matriz de features X criada: (1364, 2000)
Vetor de labels Y criado: (1364,)

Dividindo dados (80% treino / 20% teste)...


In [6]:
print("Construindo o modelo CNN (Keras)...")

# Usamos +1 no VOCAB_SIZE porque 0 é reservado para o 'padding'
model = Sequential()

# 1. Camada de Embedding
# Converte os números (1, 2, 3, 4) em vetores densos (100 dimensões)
# É como um One-Hot Encoding que "aprende"
model.add(Embedding(input_dim=VOCAB_SIZE + 1,
                    output_dim=EMBEDDING_DIM,
                    input_length=MAX_SEQ_LENGTH))

# 2. Camada Convolucional (O "Leitor" de Motivos)
# 64 = número de "filtros" (quantos motivos ele vai aprender)
# 10 = tamanho do filtro (ele vai ler "motivos" de 10 bases)
model.add(Conv1D(filters=64, kernel_size=10, activation='relu'))

# 3. Camada de Pooling (Reduz o ruído)
model.add(MaxPooling1D(pool_size=4))
model.add(Dropout(0.5)) # Dropout para evitar overfitting

# 4. Achatamento
model.add(Flatten())

# 5. Camada Densa (O "Cérebro" Classificador)
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid')) # Sigmoid para saída binária (0 ou 1)

# Compila o modelo
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

Construindo o modelo CNN (Keras)...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
print("\nIniciando treinamento da CNN...")
# (Isso vai demorar. Pegue um café.)

history = model.fit(X_train, y_train,
                    epochs=10,          # 10 "voltas" no dataset
                    batch_size=32,      # 32 genes por vez
                    validation_data=(X_test, y_test))

print("Treinamento concluído.")


Iniciando treinamento da CNN...
Epoch 1/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 117ms/step - accuracy: 0.5125 - loss: 0.7071 - val_accuracy: 0.5568 - val_loss: 0.6926
Epoch 2/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5402 - loss: 0.6915 - val_accuracy: 0.4982 - val_loss: 0.6985
Epoch 3/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5359 - loss: 0.6940 - val_accuracy: 0.5275 - val_loss: 0.6908
Epoch 4/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5531 - loss: 0.6863 - val_accuracy: 0.5201 - val_loss: 0.6964
Epoch 5/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6411 - loss: 0.6334 - val_accuracy: 0.4872 - val_loss: 0.7221
Epoch 6/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7135 - loss: 0.5891 - val_accuracy: 0.5018 - val_loss: 0.7894
Epoch 7/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7619 - loss: 0.4858 - val_accuracy: 0.4908 - val_loss: 0.8085
Epoch 8/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8799 - loss

In [8]:
print("\nAvaliando modelo nos dados de teste...")

# Precisamos do .predict() para o classification_report
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"\n--- Relatório de Desempenho (CNN) ---")
print(f"Acurácia Geral: {acc * 100:.2f}%")

print("\nRelatório de Classificação (Precisão, Recall, F1 por Classe):")
print(classification_report(y_test, y_pred))


Avaliando modelo nos dados de teste...
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step

--- Relatório de Desempenho (CNN) ---
Acurácia Geral: 54.95%

Relatório de Classificação (Precisão, Recall, F1 por Classe):
              precision    recall  f1-score   support

           0       0.56      0.49      0.52       137
           1       0.54      0.61      0.57       136

    accuracy                           0.55       273
   macro avg       0.55      0.55      0.55       273
weighted avg       0.55      0.55      0.55       273

